# Forward-pass tests for `EquivariantGNN`

Exercises `qtnet.jax_models.models_equivariant.EquivariantGNN` (the `EGNN` model
registered in `scripts/atomic/train_multitask.py`):

* one forward pass returns the expected output dict and shapes;
* `nnx.jit` compilation produces numerically identical outputs;
* full **E(3) = O(3) ⋉ ℝ³** equivariance: translations leave outputs unchanged,
  proper and improper rotations transform scalars / vectors / tensors as expected
  (scalars invariant, L=1 → `R v`, L=2 → `R T Rᵀ` in the 5-component basis).


In [ ]:
import sys, os
if os.getcwd().endswith('tests'):
    sys.path.insert(0, os.getcwd())
else:
    sys.path.insert(0, os.path.join(os.getcwd(), 'tests'))

import jax
import jax.numpy as jnp
from flax import nnx

import tests as tt
from qtnet.jax_models.models_equivariant import EquivariantGNN

print(f'JAX  version: {jax.__version__}')
print(f'JAX devices: {jax.devices()}')

runner = tt.TestRunner()


## 1. Build a small batch and an EGNN instance

The batch comes from the same synthetic pipeline the data-loading notebook uses
(8 molecules, batch size 8). The model uses a deliberately small configuration so
the cell runs in seconds.


In [ ]:
batch = tt.make_minimal_batch(n_molecules=8, batch_size=8, seed=0)
node_batch = batch.cochain_batches[0]
n_rows = int(node_batch.x.shape[0])
n_real = int(jnp.sum(node_batch.x_mask))
print(f'batch rows={n_rows}, real atoms={n_real}, padded={n_rows - n_real}')

MODEL_KW = dict(
    num_species=len(tt.ALL_ELEMENTS),
    num_node_scalars=8, num_node_vectors=4, num_node_tensors=2,
    num_edge_scalars=8, num_edge_vectors=4, num_edge_tensors=2,
    embedding_dim=16, hidden_dim=32,
    hidden_l1_channels=4, hidden_l2_channels=4,
    num_layers=2,
    geometric_filter_dim=16, geo_basis_dim=8,
    cutoff=5.0,
)
model = EquivariantGNN(**MODEL_KW, rngs=nnx.Rngs(0))
out = model(batch)
mask = out['x_mask']
print({k: tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__ for k, v in out.items()})


## 2. Output keys and shapes

The readout produces one scalar pair (N, LI), one vector channel (μ_x, μ_y, μ_z)
and one L=2 tensor (5 components). Hence `scalars` is `(n_rows, 2)`, `vectors` is
`(n_rows, 3)`, `tensors` is `(n_rows, 5)` — see
`EquivariantPerLayerReadout` defaults in `equivariant_layers.py:1594`.


In [ ]:
def _output_shape():
    expected_keys = {'scalars', 'vectors', 'tensors', 'x_mask',
                     'node_features', 'edge_features'}
    assert set(out.keys()) == expected_keys, sorted(out.keys())
    assert out['scalars'].shape == (n_rows, 2)
    assert out['vectors'].shape == (n_rows, 3)
    assert out['tensors'].shape == (n_rows, 5)
    assert out['x_mask'].shape == (n_rows,)

def _output_finite():
    for k in ('scalars', 'vectors', 'tensors'):
        assert bool(jnp.all(jnp.isfinite(out[k]))), f'{k} contains non-finite values'

runner.run('output keys & shapes', _output_shape)
runner.run('outputs are finite', _output_finite)


## 3. JIT compatibility

`nnx.jit(model)` should produce numerically identical outputs (rules out Python
branching that would silently break inside `lax.scan` / `jit`).


In [ ]:
model_jit = nnx.jit(model)
out_jit = model_jit(batch)

def _jit_matches_eager():
    for k in ('scalars', 'vectors', 'tensors'):
        err = float(jnp.max(jnp.abs(out[k] - out_jit[k])))
        assert err < 1e-4, f'{k}: jit/eager mismatch {err:.2e}'

runner.run('nnx.jit matches eager', _jit_matches_eager)

# Determinism: re-instantiating with the same seed reproduces outputs.
def _determinism():
    model_dup = EquivariantGNN(**MODEL_KW, rngs=nnx.Rngs(0))
    out_dup = model_dup(batch)
    for k in ('scalars', 'vectors', 'tensors'):
        err = float(jnp.max(jnp.abs(out[k] - out_dup[k])))
        assert err < 1e-6, f'{k}: non-deterministic ({err:.2e})'

runner.run('seed determinism', _determinism)


## 4. SO(3): proper rotation equivariance

Rotate the input batch's geometric quantities (node positions and edge gyration
tensors) by a random proper rotation `R` (det = +1) and check that:

* `out_rot['scalars'] ≈ out['scalars']` (invariant);
* `out_rot['vectors'] ≈ R · out['vectors']` (L=1 transformation);
* `out_rot['tensors'] ≈ R · out['tensors'] · Rᵀ` in the 5-component basis.

Comparisons are restricted to real (unmasked) atom rows; padded OOB rows are
unconstrained.


In [ ]:
R = tt.random_rotation_matrix(jax.random.PRNGKey(7))
assert float(jnp.linalg.det(R)) > 0.999, 'expected proper rotation'

out_rot = model(tt.rotate_complex_batch(batch, R))
result_so3 = tt.check_equivariance(
    out, out_rot, R,
    num_vector_channels=1, num_tensor_channels=1,
    name='SO(3) rotation', atol=1e-4, mask=mask,
)
print(result_so3)

runner.run('SO(3) equivariance', lambda: (_ for _ in ()).throw(AssertionError(repr(result_so3))) if not result_so3.passed else None)


## 5. Translation invariance

`EquivariantGNN` consumes positions only through pairwise differences
`pos_j − pos_i` (and gyration tensors built from those differences), so adding a
constant `t` to every real atom must leave **all** outputs unchanged.  Padded
OOB rows are deliberately not translated (they remain at zero so they stay
neutral inside `segment_sum`).


In [ ]:
t = jnp.array([1.7, -0.3, 2.5])
out_trans = model(tt.translate_complex_batch(batch, t))

def _translation_invariance():
    atol = 5e-5
    for k in ('scalars', 'vectors', 'tensors'):
        err = float(jnp.max(jnp.abs((out[k] - out_trans[k])[mask])))
        assert err < atol, f'translation broke {k}: max err {err:.2e}'
        print(f'  translation max-abs Δ{k}: {err:.2e}')

runner.run('translation invariance', _translation_invariance)


## 6. Parity (spatial inversion)

Apply `P = -I` (det = −1).  Under inversion proper vectors flip sign, proper rank-2
tensors are invariant, and gyration tensors `G = ∑ rᵢrᵢᵀ` are invariant
(`(−r)(−r)ᵀ = r rᵀ`).  Hence:

* scalars are unchanged;
* L=1 outputs flip sign;
* L=2 outputs are unchanged.

These transformations are exactly what `R = -I` produces under the generic O(3)
equations `vectors → R v` and `tensors → R T Rᵀ`.


In [ ]:
P = tt.parity_matrix()
out_par = model(tt.rotate_complex_batch(batch, P))

result_parity = tt.check_equivariance(
    out, out_par, P,
    num_vector_channels=1, num_tensor_channels=1,
    name='parity (inversion)', atol=1e-4, mask=mask,
)
print(result_parity)

# Cross-check the physics: vectors really did flip sign and tensors really stayed put.
v_diff = float(jnp.max(jnp.abs((out['vectors'] + out_par['vectors'])[mask])))
t_diff = float(jnp.max(jnp.abs((out['tensors'] - out_par['tensors'])[mask])))
print(f'  ||vectors + vectors_P||_inf = {v_diff:.2e}  (expected ~0 — vectors flip)')
print(f'  ||tensors - tensors_P||_inf = {t_diff:.2e}  (expected ~0 — tensors invariant)')

def _parity():
    assert result_parity.passed, repr(result_parity)
    assert v_diff < 1e-4, f'L=1 outputs did not flip under parity: {v_diff:.2e}'
    assert t_diff < 1e-4, f'L=2 outputs not invariant under parity: {t_diff:.2e}'

runner.run('parity equivariance', _parity)


## 7. Improper rotations (full O(3))

Random orthogonal matrices with `det(R) = -1` (proper rotation followed by a
reflection).  Together with the SO(3) test these cover all of O(3); together with
the translation test that finishes E(3).  We also sweep multiple random improper
rotations to make sure no single seed got lucky.


In [ ]:
improper_results = []
for seed in (11, 23, 37, 101):
    R_imp = tt.random_improper_rotation_matrix(jax.random.PRNGKey(seed))
    det = float(jnp.linalg.det(R_imp))
    assert det < -0.99, f'seed {seed}: expected det=-1, got {det}'
    out_imp = model(tt.rotate_complex_batch(batch, R_imp))
    res = tt.check_equivariance(
        out, out_imp, R_imp,
        num_vector_channels=1, num_tensor_channels=1,
        name=f'improper rotation (seed={seed}, det={det:+.0f})',
        atol=1e-4, mask=mask,
    )
    print(res)
    improper_results.append(res)

def _improper():
    bad = [r for r in improper_results if not r.passed]
    assert not bad, '\n'.join(repr(r) for r in bad)

runner.run('improper rotation equivariance (multi-seed)', _improper)


## 8. Control: a non-orthogonal transform must change the output

Apply a shear (not in O(3)) to confirm the equivariance check actually fires when
the symmetry is broken — without this, a model that simply ignored its inputs
would pass everything above.


In [ ]:
import dataclasses
shear = jnp.array([[1.0, 0.4, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
node_batch = batch.cochain_batches[0]
new_static = dict(node_batch.static)
new_static['pos'] = node_batch.static['pos'] @ shear.T
sheared_batch = dataclasses.replace(
    batch,
    cochain_batches=[
        dataclasses.replace(node_batch, static=new_static),
        *batch.cochain_batches[1:],
    ],
)
out_sheared = model(sheared_batch)

def _shear_changes_output():
    s_diff = float(jnp.max(jnp.abs(out['scalars'] - out_sheared['scalars'])))
    assert s_diff > 1e-3, f'shear left scalars unchanged ({s_diff:.2e}) — equivariance check is too forgiving'

runner.run('shear breaks invariance (control)', _shear_changes_output)


## Summary


In [ ]:
ok = runner.report()
assert ok, 'forward_pass.ipynb has failing tests'
